# Asteroid Orbital Visual — Automated Setup
Run each cell in order. This notebook will:
1. Flatten the raw NASA CNEOS data into a Power BI-ready CSV
2. Verify Node.js and npm are installed
3. Install `powerbi-visuals-tools` globally
4. Install the visual's npm dependencies
5. Install the pbiviz developer certificate
6. Start the pbiviz dev server (non-blocking)

In [ ]:
import subprocess, sys, os
from pathlib import Path

# Paths
NOTEBOOK_DIR  = Path(os.getcwd())
VISUAL_DIR    = NOTEBOOK_DIR / 'asteroid-orbital-visual'
DATA_SCRIPT   = NOTEBOOK_DIR / 'flatten_asteroids.py'
FLAT_CSV      = NOTEBOOK_DIR / 'asteroids_flat.csv'

print('Notebook dir :', NOTEBOOK_DIR)
print('Visual dir   :', VISUAL_DIR)
print('Data script  :', DATA_SCRIPT)
print('Python       :', sys.executable)

## Step 1 — Flatten the NASA data

In [ ]:
print('Running flatten_asteroids.py ...')
result = subprocess.run(
    [sys.executable, str(DATA_SCRIPT)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('Data prep failed — check errors above.')

rows = sum(1 for _ in open(FLAT_CSV)) - 1   # minus header
print(f'\n asteroids_flat.csv  →  {rows:,} close-approach rows')
print(f'   Size: {FLAT_CSV.stat().st_size / 1024:.1f} KB')

## Step 2 — Verify Node.js & npm

In [ ]:
def run(cmd, **kwargs):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    return r.stdout.strip() or r.stderr.strip(), r.returncode

node_ver, node_rc = run('node --version')
npm_ver,  npm_rc  = run('npm --version')

print('Node.js :', node_ver if node_rc == 0 else '❌ NOT FOUND — install from https://nodejs.org')
print('npm     :', npm_ver  if npm_rc  == 0 else '❌ NOT FOUND')

if node_rc != 0 or npm_rc != 0:
    raise EnvironmentError('Install Node.js LTS before continuing.')

## Step 3 — Install powerbi-visuals-tools globally

In [ ]:
pbiviz_ver, rc = run('pbiviz --version')
if rc == 0:
    print(f'pbiviz already installed: {pbiviz_ver}')
else:
    print('Installing powerbi-visuals-tools globally (this may take a minute)...')
    out, rc = run('npm install -g powerbi-visuals-tools')
    print(out)
    if rc != 0:
        raise RuntimeError('pbiviz install failed.')
    ver, _ = run('pbiviz --version')
    print(f'Installed pbiviz {ver}')

## Step 4 — Install visual npm dependencies

In [ ]:
print('Running npm install in visual directory...')
out, rc = run('npm install', cwd=str(VISUAL_DIR))
print(out)
if rc != 0:
    raise RuntimeError('npm install failed.')
print('\nDependencies installed.')

## Step 5 — Install the developer certificate (first time only)
> A browser window may open asking you to trust a local certificate. Click **Yes / Trust**.

In [ ]:
print('Installing pbiviz developer certificate...')
out, rc = run('pbiviz --install-cert', cwd=str(VISUAL_DIR))
print(out or 'Certificate installed (or already trusted).')

## Step 6 — Start the dev server
> The server runs in the background. Switch to Power BI Desktop and load the **Developer Visual** tile.  
> To stop it later, interrupt this kernel or run `kill_server()` in the next cell.

In [ ]:
import time, threading

server_proc = subprocess.Popen(
    'pbiviz start',
    shell=True,
    cwd=str(VISUAL_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

lines_seen = []

def _stream():
    for line in server_proc.stdout:
        lines_seen.append(line.rstrip())
        print(line, end='')

threading.Thread(target=_stream, daemon=True).start()

# Wait briefly for startup message
time.sleep(6)
started = any('listening' in l.lower() or '8080' in l for l in lines_seen)
print()
if started:
    print('✅ Dev server is running on https://localhost:8080')
    print('   Open Power BI Desktop → use the Developer Visual tile.')
else:
    print('⚠  Server may still be starting — check output above.')

In [ ]:
# Run this cell to STOP the dev server
def kill_server():
    if server_proc.poll() is None:
        server_proc.terminate()
        print('Dev server stopped.')
    else:
        print('Server already stopped.')

# kill_server()   # ← uncomment and run to stop

## Step 7 — Preview the flattened data

In [ ]:
import pandas as pd

df = pd.read_csv(FLAT_CSV)
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
# Quick summary stats
print('Hazardous asteroids :', df['potentially_hazardous'].value_counts().to_dict())
print('Unique asteroids     :', df['name'].nunique())
print('Orbiting bodies      :', df['orbiting_body'].value_counts().to_dict())
print('Date range           :', df['close_approach_date'].min(), '→', df['close_approach_date'].max())
print('\nMiss distance (AU) stats:')
print(df['miss_distance_au'].astype(float, errors='ignore').describe())